In [3]:
import pandas as pd

In [4]:
data = pd.read_csv('IMDB Dataset.csv')
print(data.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [5]:
df = pd.DataFrame(data)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
df.shape

(50000, 2)

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df["review_cleand"] = df["review"].apply(clean_text)

In [10]:
df.drop(columns=["review"], inplace=True)

In [11]:
df.head()

,sentiment,review_cleand
0,positive,one of the other reviewers has mentioned that ...
1,positive,a wonderful little production the filming te...
2,positive,i thought this was a wonderful way to spend ti...
3,negative,basically theres a family where a little boy j...
4,positive,petter matteis love in the time of money is a ...


In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vishn\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vishn\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
# create objects
stop_words = set(stopwords.words('english'))
stop_words.remove('not')
stemmer = PorterStemmer()

# function
def process_text(text):
    
    # 1. tokenization
    tokens = word_tokenize(text)
    
    # 2. remove stopwords
    filtered = [word for word in tokens if word not in stop_words]
    
    # 3. stemming
    stemmed = [stemmer.stem(word) for word in filtered]
    
    return " ".join(stemmed)

# apply to dataset
df['processed_review'] = df['review_cleand'].apply(process_text)

# check result
print(df[['review_cleand', 'processed_review']].head())

                                       review_cleand  \
0  one of the other reviewers has mentioned that ...   
1  a wonderful little production   the filming te...   
2  i thought this was a wonderful way to spend ti...   
3  basically theres a family where a little boy j...   
4  petter matteis love in the time of money is a ...   

                                    processed_review  
0  one review mention watch oz episod youll hook ...  
1  wonder littl product film techniqu unassum old...  
2  thought wonder way spend time hot summer weeke...  
3  basic there famili littl boy jake think there ...  
4  petter mattei love time money visual stun film...  


In [14]:
df["processed_review"]

0        one review mention watch oz episod youll hook ...
1        wonder littl product film techniqu unassum old...
2        thought wonder way spend time hot summer weeke...
3        basic there famili littl boy jake think there ...
4        petter mattei love time money visual stun film...
                               ...                        
49995    thought movi right good job wasnt creativ orig...
49996    bad plot bad dialogu bad act idiot direct anno...
49997    cathol taught parochi elementari school nun ta...
49998    im go disagre previou comment side maltin one ...
49999    one expect star trek movi high art fan expect ...
Name: processed_review, Length: 50000, dtype: object

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [16]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [17]:
df.sample(5)

,sentiment,review_cleand,processed_review
36513,0,this movie could have been great it is not in ...,movi could great not opinion storylin fragment...
35614,0,i enjoyed the prequels and found the relations...,enjoy prequel found relationship tucker chan p...
31035,1,a friend of mine recommended this movie citing...,friend mine recommend movi cite vocal inflect ...
47528,1,xaviera french student moves into an apartment...,xaviera french student move apart barcelona ca...
34210,1,edward furlong and christina ricci are an exce...,edward furlong christina ricci excel coupl dem...


In [18]:
# TF-IDF
# -----------------------------
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['processed_review'])
y = df['sentiment']

# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# LOGISTIC REGRESSION
# -----------------------------
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

# -----------------------------
# NAIVE BAYES
# -----------------------------
nb_model = MultinomialNB()   
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))

# -----------------------------
# TEST YOUR OWN INPUT
# -----------------------------
test_text = ["this movie is amazing"]

test_text = [process_text(clean_text(t)) for t in test_text]
test_X = vectorizer.transform(test_text)  

print("Prediction (LR):", lr_model.predict(test_X))
print("Prediction (NB):", nb_model.predict(test_X))

Logistic Regression Accuracy: 0.8853
Naive Bayes Accuracy: 0.8482
Prediction (LR): [1]
Prediction (NB): [1]


In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [20]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, lr_pred)


In [21]:
import pickle

pickle.dump(nb_model, open("model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))